In [1]:
# =================================================================
# ISTM_6217: Advanced Traffic Pattern Analysis
# Project: IoT-Driven Traffic Analytics for Congestion Prediction and
#          Optimization in New York City
# =================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# -----------------------------------------------------------------
# 1. LOAD THE PREPROCESSED DATA
# -----------------------------------------------------------------
# Ensure "Traffic_Volume_Counts_preprocessed.csv" is uploaded to Colab.
file_name = '/content/Traffic_Volume_Counts_preprocessed.csv'
df = pd.read_csv(file_name)

# -----------------------------------------------------------------
# 2. SELECT FEATURES FOR AI MODELING
# -----------------------------------------------------------------
# We identify the 24 hourly columns to use as features for Clustering.
# Note: NYC Open Data often has slight variations in column spacing.
hourly_columns = [
    '12:00-1:00 AM', '1:00-2:00AM', '2:00-3:00AM', '3:00-4:00AM', '4:00-5:00AM',
    '5:00-6:00AM', '6:00-7:00AM', '7:00-8:00AM', '8:00-9:00AM', '9:00-10:00AM',
    '10:00-11:00AM', '11:00-12:00PM', '12:00-1:00PM', '1:00-2:00PM', '2:00-3:00PM',
    '3:00-4:00PM', '4:00-5:00PM', '5:00-6:00PM', '6:00-7:00PM', '7:00-8:00PM',
    '8:00-9:00PM', '9:00-10:00PM', '10:00-11:00PM', '11:00-12:00AM'
]

# Fill any missing traffic values with 0 (Standard data cleaning)
df[hourly_columns] = df[hourly_columns].fillna(0)

# -----------------------------------------------------------------
# 3. STANDARDIZATION (Scaling)
# -----------------------------------------------------------------
# K-Means calculates distance between points. Since traffic volumes
# can vary wildly, we scale the data so each hour has equal weight.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[hourly_columns])
print("Step 3: Data scaled for K-Means.")

# -----------------------------------------------------------------
# 4. K-MEANS CLUSTERING (k=4)
# -----------------------------------------------------------------
# We increase the granularity to 4 clusters to capture more nuanced behaviors.
k = 4
kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)
print(f"Step 4: K-Means identified {k} distinct road profiles.")

# -----------------------------------------------------------------
# 5. Z-SCORE ANOMALY DETECTION
# -----------------------------------------------------------------
# We analyze 'Total_Daily_Traffic' to find statistical outliers.
# We group by 'SegmentID' to find days that are weird for THAT specific road.
print("Step 5: Detecting anomalies relative to individual road averages...")

def calculate_zscore(group):
    if group.std() == 0:
        return np.zeros(len(group))
    return (group - group.mean()) / group.std()

df['Z_Score'] = df.groupby('SegmentID')['Total_Daily_Traffic'].transform(calculate_zscore)
df['is_anomaly'] = np.abs(df['Z_Score']) > 3 # Absolute Z-score > 3 is our threshold

# -----------------------------------------------------------------
# 6. VISUALIZING RESULTS
# -----------------------------------------------------------------

# Visual 1: The 4 Traffic Profiles
plt.figure(figsize=(14, 6))
# Plotting the mean volume per hour for each cluster
cluster_trends = df.groupby('Cluster')[hourly_columns].mean().T
sns.lineplot(data=cluster_trends, palette='bright', linewidth=3)

plt.title(f'Business Insight: 4 Distinct Traffic Patterns Found', fontsize=16)
plt.ylabel('Average Vehicle Count')
plt.xlabel('Time of Day (24hr)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.legend([f'Cluster {i}' for i in range(k)], title='Road Behavior Type')
plt.show()

# Visual 2: Anomaly Scatter Plot
plt.figure(figsize=(14, 6))
sns.scatterplot(data=df, x=df.index, y='Total_Daily_Traffic', hue='is_anomaly',
                palette={True: 'red', False: 'skyblue'}, alpha=0.6)
plt.title('Business Risk: Anomalous Traffic Days (Z > 3)', fontsize=16)
plt.xlabel('Record Index')
plt.ylabel('Total Daily Volume')
plt.show()

# Summary Output
print("\n" + "="*30)
print("   EXECUTIVE AI SUMMARY")
print("="*30)
print(f"Total Traffic records: {len(df)}")
print(f"Anomalies detected:    {df['is_anomaly'].sum()}")
print("\nObservations per Cluster:")
print(df['Cluster'].value_counts().sort_index())

FileNotFoundError: [Errno 2] No such file or directory: '/content/Traffic_Volume_Counts_preprocessed.csv'